# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


## 0. Import required modules

In [16]:
import os
import sys
import pandas as pd
import spotpy as sp
from pathlib import Path
import shutil
import re

# Ensure the PWA project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

import matplotlib.pyplot as plt
import numpy as np

## 1. File and folder setup before calibration

### 1.1 Create calibration folder

In [2]:
# Set watershed name
WATERSHED_NAME = input("Enter watershed name as it appears in your naming from notebooks 0-2 (e.g., 'Manning'): ")

# Path for this notebook
NOTEBOOK_PATH = Path.cwd()

# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())

# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"

# Path for general calibration data folder
CAL_DATA_PATH = BS_DATA_PATH + r"/Calibration/"

# Path for watershed-specific calibration folder
CAL_WS_PATH = CAL_DATA_PATH + f"/{WATERSHED_NAME}/"

# Create watershed-specific calibration data folder if it doesn't exist
os.makedirs(CAL_WS_PATH, exist_ok=True)

### 1.2 Create subfolders in calibration folder

In [3]:
# Specify paths for input, output, and template folders within the calibration data folder
CAL_INPUT_PATH = CAL_WS_PATH + r"/input/"
CAL_OUTPUT_PATH = CAL_WS_PATH + r"/output/"
CAL_TPL_PATH = CAL_WS_PATH + r"/templates/"

# Create subfolders if they don't exist
SUBFOLDERS_LIST = [CAL_INPUT_PATH, CAL_OUTPUT_PATH, CAL_TPL_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Add files to appropriate locations

#### 1.3.1 Raven.exe

As per the instructions in the ```2_raven_input_gen.ipynb``` notebook, the following code expects a Raven executable file ```Raven.exe``` at ```Data/[watershed/model name]/Raven/Processed```. Please double-check before proceeding.

In [4]:
RAVEN_EXE_PATH = BS_DATA_PATH + f"/{WATERSHED_NAME}/Raven/Processed/Raven.exe"

# Move Raven.exe to CAL_WS_PATH if not already present
if not os.path.isfile(CAL_WS_PATH + "/Raven.exe"):
    if os.path.isfile(RAVEN_EXE_PATH):
        shutil.copy(RAVEN_EXE_PATH, CAL_WS_PATH + "/Raven.exe")
        print(f"Copied Raven.exe to {CAL_WS_PATH}")
    else:
        print(f"Raven.exe not found at {RAVEN_EXE_PATH}. Please ensure it is available before proceeding.")

#### 1.3.2 Depression depths file

The following code assumes that the depression depths files are all named ```CLRH_basins_depression_depths``` and currently located in the ```Data/{WATERSHED_NAME}/DepressionDepths/Processed``` folder after running the ```0_depression_depths_calc.ipynb``` notebook.

In [5]:
# Path to the folder containing the depression depths files
DD_PROCESSED_PATH = BS_DATA_PATH + f"/{WATERSHED_NAME}/DepressionDepths/Processed/"

# List of depression depths files to be copied
DD_FILES = [f for f in os.listdir(DD_PROCESSED_PATH) if f.endswith(('.cpg', '.dbf', '.prj', '.shp', '.shx'))]

# Copy each depression depths file to the calibration input folder
for dd_file in DD_FILES:
    src_file_path = os.path.join(DD_PROCESSED_PATH, dd_file)
    dest_file_path = os.path.join(CAL_INPUT_PATH, dd_file)
    shutil.copy(src_file_path, dest_file_path)
    print(f"Copied {dd_file} to {CAL_INPUT_PATH}")

Copied CLRH_basins_depression_depths.cpg to c:/Users/iyaktubay/PWA/Data//Calibration//Cypress//input/
Copied CLRH_basins_depression_depths.dbf to c:/Users/iyaktubay/PWA/Data//Calibration//Cypress//input/
Copied CLRH_basins_depression_depths.prj to c:/Users/iyaktubay/PWA/Data//Calibration//Cypress//input/
Copied CLRH_basins_depression_depths.shp to c:/Users/iyaktubay/PWA/Data//Calibration//Cypress//input/
Copied CLRH_basins_depression_depths.shx to c:/Users/iyaktubay/PWA/Data//Calibration//Cypress//input/


#### 1.3.3 GridWeights.txt file

The following code assumes that the ```GridWeights.txt``` file is currently in the ```Data/{WATERSHED_NAME}/[Runoff OR Snowmelt OR Rainfall]/Processed``` folder after running the ```1_nc_processing.ipynb``` notebook.

In [ ]:
NETCDF_CATEGORY = input("Enter the NetCDF data category used (Runoff, Snowmelt, or Rainfall): ")

# Path to the folder containing the GridWeights.txt file
NC_PROCESSED_PATH = BS_DATA_PATH + f"/{WATERSHED_NAME}/{NETCDF_CATEGORY}/Processed/"

# Path to the GridWeights.txt file
GRID_WEIGHTS_FILE = NC_PROCESSED_PATH + r"GridWeights.txt"

# Copy GridWeights.txt to CAL_INPUT_PATH if not already present
if not os.path.isfile(CAL_INPUT_PATH + "/GridWeights.txt"):
    if os.path.isfile(GRID_WEIGHTS_FILE):
        shutil.copy(GRID_WEIGHTS_FILE, CAL_INPUT_PATH + "/GridWeights.txt")
        print(f"Copied GridWeights.txt to {CAL_INPUT_PATH}")
    else:
        print(f"GridWeights.txt not found at {GRID_WEIGHTS_FILE}. Please ensure it is available before proceeding.")

#### 1.3.4 .rvp, .rvh, .rvi, .rvt, .rvc files

The following code assumes that:
- all calibration scripts will be looking for .rvp, .rvh, .rvi, and .rvt files in the ```Data/Calibration/{WATERSHED_NAME}/``` folder during calibration  
- and these files are currently in the ```Data/{WATERSHED_NAME}/Raven/Processed``` folder after running the ```2_raven_input_gen.ipynb``` notebook.

In [19]:
# Path to the folder containing the .rvp, .rvh, .rvi, .rvt, .rvc files
RA_PROCESSED_PATH = BS_DATA_PATH + f"/{WATERSHED_NAME}/Raven/Processed/"

# List of calibration files to be copied
RA_CAL_FILES = [f for f in os.listdir(RA_PROCESSED_PATH) if f.endswith(('.rvp', '.rvh', '.rvi', '.rvt', '.rvc'))]

# Copy calibration files to CAL_WS_PATH if not already present
for file_name in RA_CAL_FILES:
    src_file = RA_PROCESSED_PATH + file_name
    dest_file = CAL_WS_PATH + "/" + file_name
    if not os.path.isfile(dest_file):
        shutil.copy(src_file, dest_file)
        print(f"Copied {file_name} to {CAL_WS_PATH}")

        # If the file is an .rvt file, edit it to update .nc file name
        if file_name.endswith('.rvt'):
            with open(dest_file, "r", encoding="utf-8") as f:
                content = f.read()
            # Add "_adj" before the .nc extension (only if not already present)
            updated_content = re.sub(r"(\.nc)(?!_adj)", r"_adj\1", content)

            # Only write back if changes were made
            if updated_content != content:
                with open(dest_file, "w", encoding="utf-8") as f:
                    f.write(updated_content)
                print(f"Updated .nc references in {file_name} to include '_adj'")
            else:
                print(f"No .nc references updated in {file_name}")

#### 1.3.5 Template files
The following code assumes that all template files are currently in the ```Data/{WATERSHED_NAME}/Raven/Processed``` folder after running the ```2_raven_input_gen.ipynb``` notebook.

In [ ]:
# List of all template files to be copied
RA_TPL_FILES = [f for f in os.listdir(RA_PROCESSED_PATH) if f.endswith('tpl')]

# Copy template files to CAL_TPL_PATH if not already present
for tpl_file in RA_TPL_FILES:
    src_tpl_file = RA_PROCESSED_PATH + tpl_file
    dest_tpl_file = CAL_TPL_PATH + tpl_file
    if not os.path.isfile(dest_tpl_file):
        shutil.copy(src_tpl_file, dest_tpl_file)
        print(f"Copied {tpl_file} to {CAL_TPL_PATH}")

# Copy the netCDF .nc file to CAL_INPUT_PATH with "_tpl" at the end, if not already present
NC_FILES = [f for f in os.listdir(NC_PROCESSED_PATH) if f.endswith('.nc')]
for nc_file in NC_FILES:
    src_nc_file = NC_PROCESSED_PATH + nc_file
    dest_nc_file = CAL_INPUT_PATH + "/" + nc_file

    # Add "_adj" before the .nc extension before copying the file
    base, ext = os.path.splitext(dest_nc_file)
    new_dest_nc_file = base + "_tpl" + ext

    if not os.path.isfile(new_dest_nc_file):
        shutil.copy(src_nc_file, new_dest_nc_file)
        print(f"Copied template version of {nc_file} to {new_dest_nc_file}")


#### 1.3.6 Other files (aspect ratio, hru areas, wsc outlet, tp & tc)
The following code assumes that:
- the aspect ratios file ```aspect_ratios.csv``` is located in ```Data/{WATERSHED_NAME}/DepressionDepths/Processed``` after running the ```0_depression_depths_calc.ipynb``` notebook,
- and the HRU areas, tp & tc, and WSC outlet files are all located in ```Data/{WATERSHED_NAME}/Raven/Processed``` after running the ```2_raven_input_gen.ipynb``` notebook.

In [ ]:
# Path to aspect ratios file
ASPECT_RATIOS_FILE = DD_PROCESSED_PATH + "aspect_ratios.csv"

# List containing other file names (HRU areas, tp & tc, WSC outlet)
OTHER_FILES = [f for f in os.listdir(RA_PROCESSED_PATH) if f.endswith(('_HRU_areas.csv', 
                                                                       '_tp_tc.csv',
                                                                       '_WSC_outlet.txt'))]

# Copy aspect ratios file to CAL_INPUT_PATH if not already present
if not os.path.isfile(CAL_INPUT_PATH + "aspect_ratios.csv"):
    if os.path.isfile(ASPECT_RATIOS_FILE):
        shutil.copy(ASPECT_RATIOS_FILE, CAL_INPUT_PATH + "aspect_ratios.csv")
        print(f"Copied aspect_ratios.txt to {CAL_INPUT_PATH}")
    else:
        print(f"aspect_ratios.txt not found at {ASPECT_RATIOS_FILE}. Please ensure it is available before proceeding.")


# Copy other files to CAL_INPUT_PATH if not already present
for other_file in OTHER_FILES:
    src_other_file = RA_PROCESSED_PATH + other_file
    dest_other_file = CAL_INPUT_PATH + other_file
    if not os.path.isfile(dest_other_file):
        shutil.copy(src_other_file, dest_other_file)
        print(f"Copied {other_file} to {CAL_INPUT_PATH}")

In [41]:
# DATA_CAL = "../Data/Calibration/cypress"
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = model_str
# raven_name = "CypressRiverT_ERA5"

In [42]:
# Set calibration directory
# DATA_CAL_NEW = os.path.join(prj_path, r"Data\Calibration", raven_name).replace('\\','/')

DATA_CAL_NEW = CAL_WS_PATH
# Specify the calibration results file name and location (no .csv extension)
cal_data_path = DATA_CAL + "../Results/rope_rdrs_offset_20251009_14"

In [ ]:
# Specify necessary functions

def generate_param_tuples(spotpy_results, n_params = 7):
    # Generate tuples for parameter pairwise plots
    if isinstance(spotpy_results, pd.DataFrame):
        parameters = spotpy_results.columns[1:n_params + 1]
    else:
        parameters = spotpy_results.dtype.names[1:n_params + 1]
    parameter_tuples = [(parameters[i], parameters[j]) for i in range(len(parameters)) for j in range(i + 1, len(parameters))]
    return parameter_tuples

def plot_param_dist(_data, _param, subset_name):
    # Plot distributions of calibration parameters
    plt.hist(_data[_param], bins=10, alpha=0.7, label=f'count {_param}')
    plt.legend()
    plt.title(f'Distribution of {_param} in top 500 of the {subset_name} distribution')
    median_col = np.round(np.median(_data[_param]), 4)
    mean_col = np.round(np.mean(_data[_param]), 4)
    plt.xlabel(f'median: {median_col}, mean: {mean_col}')
    plt.show()

def sensitivity_plot(_data, x_str, y_str, objective_str, filename = None):
    # Plot objective function values (color) against 2 calibration parameters
    
    # Extract the first 3 columns from _data
    x = _data[x_str]
    y = _data[y_str]
    objective = _data[objective_str]

    # Normalize objective for color mapping
    norm_objective = (objective - objective.min()) / (objective.max() - objective.min())

    # Create a color map: blue for low objective, red for high objective
    colors = plt.cm.coolwarm(norm_objective)

    plt.figure(figsize=(8,6))
    scatter = plt.scatter(x, y, c=colors, s=40, edgecolor='k')
    plt.xlabel(x_str)
    plt.ylabel(y_str)
    plt.title(f'FAST Results: {x_str} vs {y_str} colored by {objective_str}')

    # Add colorbar to show objective scale
    sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=plt.Normalize(vmin=objective.min(), vmax=objective.max()))
    sm.set_array([])
    # cbar = plt.colorbar(sm)
    # cbar.set_label('objective')

    if filename:
        plt.savefig(filename)
        print(f"Plot saved as {filename}")

    plt.show()

## Raven test run

Test the Raven script in the main calibration path

In [ ]:
## Checking end of nc timeseries for RDRS

# from netCDF4 import Dataset

# nc_file = Dataset(DATA_CAL+"input/rdrs_runoff_wlat" + ".nc", "r")

# nc_file.variables["time"][:].min() / 8766

# nc_file.variables["time"][:].max() / 8766


In [ ]:
# raven_name = input("Enter the Raven model name (if different):")
raven_name = WATERSHED_NAME
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' and test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_name, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

## Calibration set-up and test run

Test Calibration set-up (without mpi parallelization)

In [ ]:

calibrator = spot_setup_rvn(raven_name, DATA_CAL_NEW, obj_fn= "nse_pbias", split_areas = "none", \
                            spring_eval= True, low_seep=False, routing=True, dep_scaling=True, runoff_scaling=True, channel_scaling = True,
                            mpi_cores=0)


In [ ]:
# Use the Pareto Archived Dynamically Dimensioned Search (PADDS) algorithm for calibration
# sampler=sp.algorithms.padds(spot_setup=calibrator, # Core model setup object
#                             dbname = "padds_raven", # Save results of cal run to this database
#                             dbformat='csv')

 
sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
                           parallel='seq',
                           dbname='test',
                           dbformat='csv')

# sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
#                            parallel='seq',
#                            dbname='mc_raven',
#                            dbformat='csv')



In [ ]:
rep = 3
sampler.sample(rep)

## REMINDER - try to reduce the evaluation period to 2003 onward to improve performance

# setup = calibrator  # your spot_setup object

# params = [p.optguess for p in setup.parameters]
# print("params:", params)

# sim = setup.simulation(params)
# evals = setup.evaluation()
# obj = setup.objectivefunction(sim, evals)
# print("Objective output:", obj)

## FAST Analysis

Run fast method in mpiexec

With 5 parameters, the number of reps required for FAST is 2245

In [ ]:
# Load FAST results and plot sensitivity

fast_results = sp.analyser.load_csv_results(cal_data_path)
sp.analyser.plot_fast_sensitivity(fast_results, number_of_sensitiv_pars=3)

In [ ]:
fast_results = fast_results[['like1', 'pardep_fact', 'parseep_k', 'part_peak_fact']]

parameter_tuples = generate_param_tuples(fast_results, n_params = 7)

for param1, param2 in parameter_tuples:
    sensitivity_plot(fast_results, param1, param2, 'like1') # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">

## Calibration Analysis

### Define calibration subset

In [ ]:
calibration_results = sp.analyser.load_csv_results(cal_data_path)

In [ ]:
# Only relevant for a converged calibration
# Compare the prior (first 2500) and posterior (last 2500) distributions of the Spotpy results

calibration_prior = calibration_results[:2500]
calibration_post = calibration_results[-2500:]

plot_param_dist(calibration_prior, 'like1', 'prior')
plot_param_dist(calibration_post, 'like1', 'post')

In [ ]:
# Some options for selecting the best results for analysis

# calibration_post_best = pd.DataFrame(calibration_results[7500:]).sort_values(by='like1', ascending=False).head(500)

calibration_post_best = pd.DataFrame(calibration_post).sort_values(by='like1', ascending=False).head(100)

# calibration_post_best = pd.DataFrame(calibration_results).sort_values(by='like1', ascending=False).head(200)

### Evaluate prior or post subsets

In [ ]:
# Rename the objective function column for clarity
obj_fn_str = "nse-1-norm(pbias)"
calibration_post_best.rename(columns = {"like1" : obj_fn_str}, inplace=True)

def plot_calibration_distributions(calibration_dataset: pd.DataFrame):
    """
    Plots histograms for each column in the calibration dataset,
    showing the distribution, median, and mean for each parameter.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    for col in calibration_dataset.columns:
        plt.hist(calibration_dataset[col], bins=10, alpha=0.7, label=f'count {col}')
        plt.legend()
        plt.title(f'Distribution of {col} in top {len(calibration_dataset)} of the final ROPE simulations')
        median_col = np.round(calibration_dataset[col].median(), 4)
        mean_col = np.round(calibration_dataset[col].mean(), 4)
        plt.xlabel(f'median: {median_col}, mean: {mean_col}')
        # plt.savefig(f"post_{col}.jpg")
        plt.show()

plot_calibration_distributions(pd.DataFrame(calibration_post))

In [ ]:
parameter_tuples = generate_param_tuples(calibration_post_best, n_params = 8)

# View correlations between parameters in the best results
for param1, param2 in parameter_tuples:
        pearson_corr = calibration_post_best[[param1, param2]].corr().iloc[0, 1]
        if np.abs(pearson_corr) > 0.3:
            print(f'Parameter 1: {param1}, Parameter 2: {param2}, corr:{pearson_corr}')

In [ ]:
# Generate heat bi-plots of best results
for param1, param2 in parameter_tuples:
    sensitivity_plot(calibration_post_best, param1, param2, 'nse-1-norm(pbias)') # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">